In [49]:

import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


#Data Loading

In [50]:
df=pd.read_csv("movie dataset.csv")

In [51]:
df.head()

,genres,keywords,title,overview,title.1
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",Avatar,"In the 22nd century, a paraplegic Marine is di...",Avatar
1,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End
2,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",Spectre,A cryptic message from Bond’s past sends him o...,Spectre
3,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",The Dark Knight Rises,Following the death of District Attorney Harve...,The Dark Knight Rises
4,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",John Carter,"John Carter is a war-weary, former military ca...",John Carter


#Data Cleaning and Preprocessing

In [52]:

def convert_keywords(keyword):
    try:
        data = ast.literal_eval(keyword)
        return " ".join(item["name"] for item in data)
    except (ValueError, SyntaxError, TypeError):
        return ""

df["keywords"] = df["keywords"].apply(convert_keywords)

In [53]:
df.head()

,genres,keywords,title,overview,title.1
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",culture clash future space war space colony so...,Avatar,"In the 22nd century, a paraplegic Marine is di...",Avatar
1,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",ocean drug abuse exotic island east india trad...,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End
2,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",spy based on novel secret agent sequel mi6 bri...,Spectre,A cryptic message from Bond’s past sends him o...,Spectre
3,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",dc comics crime fighter terrorist secret ident...,The Dark Knight Rises,Following the death of District Attorney Harve...,The Dark Knight Rises
4,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",based on novel mars medallion space travel pri...,John Carter,"John Carter is a war-weary, former military ca...",John Carter


In [54]:
def convert_genres(genres):
    try:
        data = ast.literal_eval(genres)
        return " ".join(item["name"] for item in data)
    except (ValueError, SyntaxError, TypeError):
        return ""

df["genres"] = df["genres"].apply(convert_genres)

In [55]:
df.head()

,genres,keywords,title,overview,title.1
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Avatar,"In the 22nd century, a paraplegic Marine is di...",Avatar
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End
2,Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...,Spectre,A cryptic message from Bond’s past sends him o...,Spectre
3,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,The Dark Knight Rises,Following the death of District Attorney Harve...,The Dark Knight Rises
4,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,John Carter,"John Carter is a war-weary, former military ca...",John Carter


In [56]:
df=df.drop("title.1",axis=1)

In [57]:
df.head()

,genres,keywords,title,overview
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...,Spectre,A cryptic message from Bond’s past sends him o...
3,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,The Dark Knight Rises,Following the death of District Attorney Harve...
4,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,John Carter,"John Carter is a war-weary, former military ca..."


#Feature Engineering

In [58]:
df["overview"] = df["overview"].fillna("")

df["tags"] = (
    df["genres"].fillna("") + " " +
    df["keywords"].fillna("") + " " +
    df["overview"]
)

print(df[["title", "tags"]].head())

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                                tags  
0  Action Adventure Fantasy Science Fiction cultu...  
1  Adventure Fantasy Action ocean drug abuse exot...  
2  Action Adventure Crime spy based on novel secr...  
3  Action Crime Drama Thriller dc comics crime fi...  
4  Action Adventure Science Fiction based on nove...  


In [59]:
df.head()

,genres,keywords,title,overview,tags
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Avatar,"In the 22nd century, a paraplegic Marine is di...",Action Adventure Fantasy Science Fiction cultu...
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",Adventure Fantasy Action ocean drug abuse exot...
2,Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...,Spectre,A cryptic message from Bond’s past sends him o...,Action Adventure Crime spy based on novel secr...
3,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,The Dark Knight Rises,Following the death of District Attorney Harve...,Action Crime Drama Thriller dc comics crime fi...
4,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,John Carter,"John Carter is a war-weary, former military ca...",Action Adventure Science Fiction based on nove...


#Text Preprocessing

In [60]:
df["tags"] = df["tags"].str.lower().str.strip()

In [67]:
df.head()

,genres,keywords,title,overview,tags
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Avatar,"In the 22nd century, a paraplegic Marine is di...",action adventure fantasy science fiction cultu...
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",adventure fantasy action ocean drug abuse exot...
2,Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...,Spectre,A cryptic message from Bond’s past sends him o...,action adventure crime spy based on novel secr...
3,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,The Dark Knight Rises,Following the death of District Attorney Harve...,action crime drama thriller dc comics crime fi...
4,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,John Carter,"John Carter is a war-weary, former military ca...",action adventure science fiction based on nove...


#Feature Extraction Using TF-IDF

In [62]:
tfidf=TfidfVectorizer(stop_words="english")
vectors = tfidf.fit_transform(df["tags"])

In [63]:
print(vectors.shape)

(4803, 23004)


#Similarity Calculation

In [64]:
similarity = cosine_similarity(vectors)

print(similarity.shape)

(4803, 4803)


#Recommendation System Development

In [65]:
def recommend(movie):
    movie = movie.lower()

    matches = df[df["title"].str.lower() == movie]

    if matches.empty:
        return "Movie not found!"

    index = matches.index[0]

    scores = list(enumerate(similarity[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommended_movies = []

    for i, score in scores[1:6]:
        recommended_movies.append(df.loc[i, "title"])

    return recommended_movies

#Testing and Validation

In [66]:
print(recommend("Superman Returns"))

['Superman II', 'Superman', 'Superman IV: The Quest for Peace', 'Superman III', 'Man of Steel']
